<style>
    @import url('https://fonts.googleapis.com/css2?family=Oswald:wght@400;600&display=swap');
    h1.course-title {
        font-family: 'Oswald', sans-serif;
        font-size: 2.4em;
        color: #E7C173;
        letter-spacing: 0.05em;
        border-bottom: 2px solid #E7C173;
        padding-bottom: 0.3em;
        margin-bottom: 0.2em;
    }
    h2.course-subtitle { font-family: 'Oswald', sans-serif; color: #aaa; font-size: 1.2em; }
</style>

<h1 class='course-title'>MACHINE LEARNING IN INDUSTRY</h1>
<h2 class='course-subtitle'>Cardo AI · MSCA Digital Doctoral Network · Day 4 Workshop</h2>

# Day 4 Workshop — MLOps Hands-On Exercise

**Time:** 1:30–3:00 PM (90 minutes for Steps 1–3; Step 4 is a stretch goal)

## What you will do

| Step | Task | Reference |
|---|---|---|
| **1** | Log a training run to MLflow and register the model | `notebooks/01_mlflow_tracking.ipynb` |
| **2** | Run NannyML drift detection and log the report | `notebooks/02_nannyml_monitoring.ipynb` |
| **3** | Build the Docker image and query the prediction API | `day4/Dockerfile` + `day4/README.md` |
| **4** *(stretch)* | Trigger the GitHub Actions CI/CD workflow on a fork | `.github/workflows/ml-pipeline.yml` |

**Rules:**
- Each `TODO` has a reference comment pointing to the relevant notebook section.
- The skeleton runs without errors even with TODOs left blank — you add functionality incrementally.
- Work in your group. Discuss before you type.

> 💡 **Tip:** If you get stuck, the complete solution lives in the lecture notebooks and `day4/src/`. Use them — that's what they're there for.

---
## Setup — Run this first (do not modify)

In [1]:
# ── All imports pre-filled — no time wasted here ─────────────────────────────
import sys, time, json, tempfile
from pathlib import Path

import mlflow
import mlflow.sklearn
import nannyml as nml
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

repo_root = Path.cwd().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from day4.src.train import (
    DATA_PATH, SEED, TARGET_BIN_COL,
    build_pipeline, get_feature_columns, load_data, split_data,
)

np.random.seed(SEED)
print("Setup complete.")

Setup complete.


In [ ]:
# ── Data loading — pre-filled, just run it ────────────────────────────────────
data_path = repo_root / DATA_PATH
df = load_data(data_path)
train_df, val_df, test_df = split_data(df)

numeric_cols, categorical_cols = get_feature_columns(train_df)
feature_cols = numeric_cols + categorical_cols

X_train, y_train = train_df[feature_cols], train_df[TARGET_BIN_COL]
X_val,   y_val   = val_df[feature_cols],   val_df[TARGET_BIN_COL]

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(test_df)} rows")
print(f"Features: {len(feature_cols)}")

Train: 5909 | Val: 1488 | Test: 1874 rows
Features: 19


---
## Step 1 — Experiment Tracking with MLflow

**Goal:** Log a training run (params, metrics, model) to a local MLflow server. Register the model.

**Prerequisite:** Start the MLflow server in a separate terminal:
```bash
make -f day4/Makefile mlflow-server
```

> **Reference:** `notebooks/01_mlflow_tracking.ipynb`, Sections 4–7

In [ ]:
# ── 1a. Configure MLflow ──────────────────────────────────────────────────────

# TODO: Set the tracking URI to http://127.0.0.1:5000
# See: notebook 01, Section 4 — mlflow.set_tracking_uri(...)
mlflow.set_tracking_uri(...)  # ← your code here

# TODO: Create or select an experiment named "workshop-adult-income"
# See: notebook 01, Section 4 — mlflow.set_experiment(...)
mlflow.set_experiment(...)    # ← your code here

print(f"Tracking URI : {mlflow.get_tracking_uri()}")

In [ ]:
# ── 1b. Define hyperparameters ────────────────────────────────────────────────

# TODO: Fill in the hyperparameters.
# Use the Day 2 best config: n_estimators=300, learning_rate=0.1, max_depth=5
params = {
    "n_estimators":  ...,   # ← your code here
    "learning_rate": ...,   # ← your code here
    "max_depth":     ...,   # ← your code here
}

print("Params:", params)

In [ ]:
# ── 1c. Train and log ─────────────────────────────────────────────────────────

# TODO: Complete the MLflow run.
# Inside the with-block:
#   1. Log params with mlflow.log_params(params)
#   2. Train: pipe = build_pipeline(numeric_cols, categorical_cols, **params); pipe.fit(...)
#   3. Compute val_auc, val_f1, val_accuracy and log with mlflow.log_metrics({...})
#   4. Log the model with mlflow.sklearn.log_model(pipe, artifact_path="model")
# See: notebook 01, Section 5

workshop_run_id = None

with mlflow.start_run(run_name="workshop-run") as run:
    workshop_run_id = run.info.run_id

    # ↓↓↓ your code here ↓↓↓


    # ↑↑↑ your code here ↑↑↑
    pass  # remove this once you've added your code

print(f"Run ID: {workshop_run_id}")
print("Open http://127.0.0.1:5000 and find your run in 'workshop-adult-income'.")

In [ ]:
# ── 1d. Register the model ────────────────────────────────────────

# TODO: Register your model in the MLflow Model Registry.
# See: notebook 01, Section 7

# ↓↓↓ your code here ↓↓↓


# ↑↑↑ your code here ↑↑↑

**Step 1 self-check:**
- [ ] My run appears in the MLflow UI under experiment `workshop-adult-income`
- [ ] The run has logged params, metrics (including `val_auc`), and a model artifact
- [ ] (Optional) The model is registered in the Model Registry

---
## Step 2 — Drift Detection with NannyML

**Goal:** Build reference/analysis sets, run univariate drift detection, log the report to MLflow.

> **Reference:** `notebooks/02_nannyml_monitoring.ipynb`, Sections 5–7

In [ ]:
# ── 2a. Re-train a model (needed for scoring) ─────────────────────────────────
# Pre-filled — just run it.
default_params = {"n_estimators": 300, "learning_rate": 0.1, "max_depth": 5}
pipe = build_pipeline(numeric_cols, categorical_cols, **default_params)
pipe.fit(X_train, y_train)
print("Model trained.")

In [ ]:
# ── 2b. Build reference and analysis sets — pre-filled ────────────────────────
# You do not need to change this cell.

reference_df = df[df["split"] == "train"][feature_cols + [TARGET_BIN_COL]].copy()
reference_df["y_pred_proba"] = pipe.predict_proba(reference_df[feature_cols])[:, 1]
reference_df["y_pred"] = (reference_df["y_pred_proba"] >= 0.5).astype(int)
reference_df["timestamp"] = pd.date_range("2023-01-01", periods=len(reference_df), freq="h")

analysis_df = test_df[feature_cols].copy()
analysis_df["y_pred_proba"] = pipe.predict_proba(analysis_df)[:, 1]
analysis_df["y_pred"] = (analysis_df["y_pred_proba"] >= 0.5).astype(int)
analysis_df["timestamp"] = pd.date_range("2024-01-01", periods=len(analysis_df), freq="h")

print(f"Reference: {len(reference_df)} rows | Analysis: {len(analysis_df)} rows")

In [ ]:
# ── 2c. Univariate drift detection ────────────────────────────────────────────

# TODO: Instantiate UnivariateDriftCalculator, fit on reference_df, calculate on analysis_df.
# See: notebook 02, Section 7

# ↓↓↓ your code here ↓↓↓

univariate_results = None  # replace with your result

# ↑↑↑ your code here ↑↑↑

In [ ]:
# ── 2d. Log NannyML report to MLflow ─────────────────────────────────────────

# TODO: Log the drift report HTML as an MLflow artifact.
# See: notebook 02, Section 10

# ↓↓↓ your code here ↓↓↓


# ↑↑↑ your code here ↑↑↑

**Step 2 self-check:**
- [ ] Univariate drift plot is displayed
- [ ] I can identify which features (if any) drifted
- [ ] The HTML report is logged as an artifact in the `adult-income-monitoring` experiment

---
## Step 3 — Docker

**Goal:** Build the prediction service container and get a live prediction from it.

This step happens **in a terminal**, not in this notebook.

### Instructions

From the **repo root**:

```bash
# Build the Docker image (requires Step 1 to be complete)
make -f day4/Makefile docker-build

# Run the container
make -f day4/Makefile docker-run
```

In a **second terminal**, test it:

```bash
# Health check
curl http://localhost:8080/health

# Prediction (copy-paste this)
curl -X POST http://localhost:8080/predict \
     -H "Content-Type: application/json" \
     -d '{
           "age": 35,
           "workclass": "Private",
           "education": "Bachelors",
           "education_num": 13,
           "marital_status": "Married-civ-spouse",
           "occupation": "Prof-specialty",
           "relationship": "Husband",
           "race": "White",
           "sex": "Male",
           "capital_gain": 0,
           "capital_loss": 0,
           "hours_per_week": 45,
           "native_country": "United-States"
         }'
```

You can also explore the API docs at **http://localhost:8080/docs**.

### Questions to discuss with your group

1. The Dockerfile copies `mlruns/` into the image at build time. What are the trade-offs of this approach vs. pulling the model at container startup from a remote server?
2. The model URI uses `file:///app/mlruns`. What would you change to use the Model Registry instead?
3. The container exposes a single prediction endpoint. In production, what other endpoints would you want? (Hint: batch scoring, metrics, model version info)

In [3]:
# ── Optional: call the API from the notebook ──────────────────────────────────
# (requires the container to be running: make docker-run)
import urllib.request, urllib.error, json as _json

SAMPLE_PAYLOAD = {
    "age": 35, "workclass": "Private", "education": "Bachelors",
    "education_num": 13, "marital_status": "Married-civ-spouse",
    "occupation": "Prof-specialty", "relationship": "Husband",
    "race": "White", "sex": "Male",
    "capital_gain": 0, "capital_loss": 0,
    "hours_per_week": 45, "native_country": "United-States",
}

try:
    req = urllib.request.Request(
        "http://localhost:8080/predict",
        data=_json.dumps(SAMPLE_PAYLOAD).encode(),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=3) as resp:
        result = _json.loads(resp.read())
    print(f"Prediction : {result['label']} (p={result['probability']:.4f})")
    print(f"Model URI  : {result['model_uri']}")
except urllib.error.URLError:
    print("Container not running. Start it with: make -f day4/Makefile docker-run")

Prediction : >50K (p=0.7060)
Model URI  : /app/model


**Step 3 self-check:**
- [ ] `curl http://localhost:8080/health` returns `{"status": "ok", "model_loaded": true}`
- [ ] The `/predict` endpoint returns a JSON response with `prediction`, `probability`, and `label`
- [ ] I can explain why the model URI uses `file:///app/mlruns`

---
## Step 4 — CI/CD with GitHub Actions

**Goal:** Trigger the pre-built ML pipeline workflow on your own fork and inspect the artifacts.

This step is **observational** — no code to write. It's about understanding how the CI/CD system works.

### Instructions

1. **Fork** the course repository on GitHub.
2. **Clone** your fork locally (or push from your existing clone).
3. Make a small change to any file in `day4/` (e.g., add a comment to `train.py`) and `git push`.
4. Navigate to the **Actions** tab on your fork on GitHub.
5. Watch the `ML Pipeline` workflow run. Expand each step to see the logs.
6. Once complete, click **Summary** → download the artifacts:
   - `mlflow-run-<sha>` — the full MLflow run directory (explore it locally with `mlflow ui`)
   - `nannyml-report-<sha>` — open `cbpe_report.html` and `drift_report.html` in your browser

### Read the YAML

Open `.github/workflows/ml-pipeline.yml` and answer these questions:

1. What is `MLFLOW_TRACKING_URI` set to in the workflow? Why is this different from your local setup?
2. Why does the drift-check step use `continue-on-error: true`?
3. Which step would cause the workflow to fail (exit code 1) if drift exceeds the threshold?
4. How would you modify the workflow to also build and push the Docker image to a container registry?
5. What would you need to change to make this run on a schedule (e.g., every night at 2 AM)?

**Step 4 self-check:**
- [ ] The GitHub Actions workflow ran successfully on my fork
- [ ] I downloaded and inspected both artifact archives
- [ ] I can answer all five questions above
- [ ] I understand why the CI uses `file://./mlruns` and how a production setup would differ

---
## End-of-Workshop Checklist

Before Day 5, confirm that your group has:

- [ ] **MLflow:** At least one training run logged with params, metrics, and a model artifact
- [ ] **MLflow:** The model registered in the Model Registry (any stage)
- [ ] **NannyML:** Drift detection ran on reference/analysis sets; report logged to MLflow
- [ ] **Docker:** The container started successfully and returned a prediction from `/predict`
- [ ] **Discussion:** Your group has decided which MLOps practices you'll incorporate into your project (minimum: experiment tracking + one drift check)
- [ ] **GitHub Actions:** Workflow triggered and artifacts downloaded

---

**Project integration (3:00–4:30 PM):**

Transition to group project time. Discuss:
1. What will your reference and analysis sets be? (What's the equivalent of the train/test split in your project data?)
2. Which features are most likely to drift in your domain?
3. What will a drift alert trigger in your project? (Retrain? Alert? Manual review?)